NOTEBOOK UTILS

Carrega todas as funções necessárias para a execução das atividades principais.

# CONFIGURAÇÕES GERAIS

## Importar bibliotecas

In [0]:
from datetime import datetime
from pyspark.sql import DataFrame
import logging
import pandas as pd
from io import BytesIO
from deltalake import write_deltalake
import time
import json
from pyspark.sql.functions import (avg, current_timestamp, col,coalesce, count, datediff, desc, lit, min, regexp_replace, round, row_number, split, trim, unix_timestamp, upper, year, month, dayofmonth, hour)
from pyspark.sql.window import Window
from deltalake import DeltaTable
from pyspark.sql.functions import expr
from pyspark.sql.types import NullType, StringType

## Configuração do logging

In [0]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

log = logging.getLogger("pipeline_ingestion")

logging.getLogger("azure").setLevel(logging.WARNING)
logging.getLogger("azure.identity").setLevel(logging.WARNING)
logging.getLogger("azure.core").setLevel(logging.WARNING)

## Configurações de conexão com o storage

# FUNÇÕES DE LEITURA

## Função de leitura arquivos parquet

In [0]:
def read_parquet_to_spark_df(
    container_client : str,
    file_path: str) -> DataFrame:
    """
    Realiza o download de um arquivo Parquet armazenado no ADLS,
    converte o conteúdo para um DataFrame Spark e retorna o resultado.

    Args:
        file_path (str): Caminho completo do arquivo no ADLS.

    Returns:
        DataFrame: DataFrame Spark contendo os dados do arquivo.
    """

    data = (
        container_client
        .get_file_client(file_path)
        .download_file()
        .readall()
    )
    
    pdf = pd.read_parquet(BytesIO(data))

    return spark.createDataFrame(pdf)

## Função de leitura de tabelas delta

In [0]:
def read_delta_to_spark_df(
    path: str,
    storage_options: dict,
) -> DataFrame:
    """
    Lê uma tabela Delta do ADLS e
    retorna um DataFrame Spark.
    """

    dt = DeltaTable(
        path,
        storage_options={
            "account_name": storage_options["storage_account_name"],
            "tenant_id": storage_options["tenant_id"],
            "client_id": storage_options["client_id"],
            "client_secret": storage_options["client_secret"]
        }
    )

    pdf = dt.to_pandas()

    return spark.createDataFrame(pdf)

# FUNÇÕES AUXILIARES

## Listar arquivos do container. Opcionais: pasta e arquivo

In [0]:
def list_files(
    container_client,
    folder_name: str = None,
    file_name_contains: str = None
) -> list:
    """
    Lista arquivos do container ADLS com filtros opcionais.

    Args:
        container_client:
            Cliente do File System do ADLS.

        folder_name (str, optional):
            Caminho da pasta a ser consultada.
            Se não informado, consulta todo o container.

        file_name_contains (str, optional):
            Trecho que deve existir no nome do arquivo.
            Se não informado, retorna todos os arquivos encontrados.

    Returns:
        list:
            Lista contendo os caminhos completos dos arquivos encontrados.
    """

    files = []

    paths = (
        container_client.get_paths(folder_name)
        if folder_name
        else container_client.get_paths()
    )

    for path in paths:

        if path.is_directory:
            continue

        if file_name_contains and file_name_contains not in path.name:
            continue

        files.append(path.name)

    return files

## Converter o tipo de colunas vazias de NullType para StringType

In [0]:
def cast_nulltype_columns_to_string(df: DataFrame) -> DataFrame:
    """
    Converte colunas NullType para StringType.
    """

    for field in df.schema.fields:
        if isinstance(field.dataType, NullType):
            df = df.withColumn(
                field.name,
                lit(None).cast(StringType())
            )

    return df

# FUNÇÕES DE ESCRITA

## Escrever no SQL SERVER

In [0]:
def write_sql_server(
    df: DataFrame,
    table_name: str,
    jdbc_hostname: str,
    jdbc_database: str,
    jdbc_username: str,
    jdbc_password: str,
    mode: str = "append"
) -> None:
    """
    Escreve um DataFrame Spark em uma tabela do SQL Server.

    Args:
        df (DataFrame):
            DataFrame Spark a ser gravado.

        table_name (str):
            Nome completo da tabela destino
            (ex.: "squad2.ecommerce_clientes").

        jdbc_hostname (str):
            Host do SQL Server.

        jdbc_database (str):
            Nome do banco de dados.

        jdbc_username (str):
            Usuário de acesso ao banco.

        jdbc_password (str):
            Senha de acesso ao banco.

        mode (str, optional):
            Modo de escrita do Spark.
            Ex.: append, overwrite, ignore, errorifexists.
            Default: append.
    """

    df.write \
        .format("sqlserver") \
        .option("host", jdbc_hostname) \
        .option("port", 1433) \
        .option("database", jdbc_database) \
        .option("dbtable", table_name) \
        .option("user", jdbc_username) \
        .option("password", jdbc_password) \
        .mode(mode) \
        .save()

## Escrever no Data Lake

In [0]:
def write_adls(
    df: DataFrame,
    path: str,
    storage_options = dict,
    mode: str = "append"
) -> None:
    """
    Escreve um DataFrame Spark em formato Delta no ADLS Gen2
    utilizando deltalake-python e autenticação via Service Principal.

    Args:
        df (DataFrame):
            DataFrame Spark a ser gravado.

        path (str):
            Caminho de destino no ADLS.

        storage_account_name (str):
            Nome da Storage Account.

        tenant_id (str):
            Tenant ID do Azure AD.

        client_id (str):
            Client ID do Service Principal.

        client_secret (str):
            Client Secret do Service Principal.

        mode (str, optional):
            Modo de escrita.
            Ex.: append ou overwrite.
            Default: append.
    """

    pdf = df.toPandas()

    if df.count() == 0:
        log.warning("DataFrame vazio. Escrita ignorada.")
        return

    # Evita erro do Delta Lake quando uma coluna está totalmente nula.

    for column in pdf.columns:
        if pdf[column].isna().all():
            pdf[column] = pdf[column].astype("string")


    write_deltalake(
        table_or_uri=path,
        data=pdf,
        mode=mode,
        storage_options={
            "account_name": storage_account_name,
            "tenant_id": tenant_id,
            "client_id": client_id,
            "client_secret": client_secret
        }
    )

## Escrever no Data Lake particionando por data

In [0]:
def write_adls_partitioned(
    df: DataFrame,
    path: str,
    storage_options: dict,
    partition_by: list[str],
    mode: str = "append"
) -> None:
    """
    Escreve um DataFrame Spark em formato Delta
    particionado no ADLS Gen2.
    """

    if df.isEmpty():

        log.warning(
            "DataFrame vazio. Escrita ignorada."
        )

        return

    pdf = df.toPandas()

    write_deltalake(
        table_or_uri=path,
        data=pdf,
        mode=mode,
        partition_by=partition_by,
        storage_options={
            "account_name": storage_account_name,
            "tenant_id": tenant_id,
            "client_id": client_id,
            "client_secret": client_secret
        }
    )

# FUNÇÕES DE CONTROLE E COLUNAS DE AUDITORIA

## Carregar o arquivo de controle

In [0]:
def load_processed_files(
    container_client,
    control_file_path: str
) -> set:
    """
    Carrega os arquivos já processados a partir do arquivo
    de controle armazenado no ADLS.

    Args:
        container_client:
            Cliente do container ADLS.

        control_file_path (str):
            Caminho do arquivo JSON de controle.

            Ex.:
            control/control_file.json

    Returns:
        set:
            Conjunto contendo os caminhos dos arquivos já processados.
    """

    try:
        file_client = container_client.get_file_client(
            control_file_path
        )

        downloaded_file = file_client.download_file()

        processed_files = json.loads(
            downloaded_file.readall()
        )

        return set(processed_files)

    except Exception:
        return set()

## Salvar os arquivos processados no arquivo de controle

In [0]:
def save_processed_file(
    container_client,
    control_file_path: str,
    file_path: str
) -> None:
    """
    Adiciona um arquivo ao controle de arquivos processados
    armazenado no ADLS.

    Args:
        container_client:
            Cliente do container ADLS.

        control_file_path (str):
            Caminho do arquivo JSON de controle.

        file_path (str):
            Caminho do arquivo processado.
    """

    try:
        file_client = container_client.get_file_client(
            control_file_path
        )

        downloaded_file = file_client.download_file()

        processed_files = json.loads(
            downloaded_file.readall()
        )

    except Exception:
        processed_files = []

    if file_path not in processed_files:

        processed_files.append(file_path)

        file_client = container_client.get_file_client(
            control_file_path
        )

        file_client.upload_data(
            json.dumps(
                processed_files,
                indent=4
            ),
            overwrite=True
        )

## Adicionar colunas de auditoria na tabela SILVER

In [0]:
def add_silver_audit_columns(
    df: DataFrame
) -> DataFrame:
    """
    Adiciona metadados da Silver.
    """

    return (
        df
        .withColumn(
            "silver_processed_at",
            current_timestamp()
        )
        .withColumn(
            "processed_year",
            year("silver_processed_at")
        )
        .withColumn(
            "processed_month",
            month("silver_processed_at")
        )
        .withColumn(
            "processed_day",
            dayofmonth("silver_processed_at")
        )
        .withColumn(
            "processed_hour",
            hour("silver_processed_at")
        )
    )

## Adicionar colunas de auditoria na tabela GOLD

In [0]:
def add_gold_audit_columns(
    df: DataFrame
) -> DataFrame:
    """
    Adiciona colunas de auditoria
    para a camada Gold.

    Args:
        df (DataFrame):
            DataFrame de entrada.

    Returns:
        DataFrame:
            DataFrame enriquecido.
    """

    return (
        df
        .withColumn(
            "gold_processed_at",
            current_timestamp()
        )
        .withColumn(
            "processed_year",
            year(
                current_timestamp()
            )
        )
        .withColumn(
            "processed_month",
            month(
                current_timestamp()
            )
        )
        .withColumn(
            "processed_day",
            dayofmonth(
                current_timestamp()
            )
        )
        .withColumn(
            "processed_hour",
            hour(
                current_timestamp()
            )
        )
    )

# FUNÇÕES DE VALIDAÇÃO E QUARENTENA

## Adicionar dados na tabela de quarentena

In [0]:
def append_to_quarantine(
    df_quarantine: DataFrame | None,
    df_invalid: DataFrame
) -> DataFrame:
    """
    Adiciona registros inválidos ao DataFrame
    de quarentena.

    Args:
        df_quarantine (DataFrame | None):
            DataFrame acumulador da quarentena.

        df_invalid (DataFrame):
            Registros inválidos da regra atual.

    Returns:
        DataFrame:
            DataFrame de quarentena atualizado.
    """

    if df_invalid.count() == 0:
        return df_quarantine

    if df_quarantine is None:
        return df_invalid

    return (
        df_quarantine
        .unionByName(
            df_invalid,
            allowMissingColumns=True
        )
    )

## Adicionar metadados na tabela de quarentena

In [0]:
def add_quarantine_metadata(
    df: DataFrame,
    reason: str
) -> DataFrame:
    """
    Adiciona metadados de quarentena.

    Args:
        df (DataFrame):
            DataFrame inválido.

        reason (str):
            Motivo da rejeição.

    Returns:
        DataFrame:
            DataFrame enriquecido.
    """

    return (
        df
        .withColumn(
            "quarantine_reason",
            lit(reason)
        )
        .withColumn(
            "quarantine_created_at",
            current_timestamp()
        )
    )

# VALIDAÇÃO DAS REGRAS DE NEGÓCIO

## Validação das regras da SILVER 'clientes'

### Regra 1 	
Schema completo em cada micro-lote (id_cliente, uuid_cliente, nome, sobrenome, email, dt_cadastro)  

Colunas faltando quebram joins com pedidos.


In [0]:
def validate_clientes_schema(
    df: DataFrame
) -> None:
    """
    Valida se todas as colunas obrigatórias
    estão presentes no micro-lote.

    Args:
        df (DataFrame):
            DataFrame Spark.

    Raises:
        ValueError:
            Caso alguma coluna obrigatória
            esteja ausente.
    """

    required_columns = [
        "id_cliente",
        "uuid_cliente",
        "nome",
        "sobrenome",
        "email",
        "dt_cadastro"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            "Schema inválido. "
            f"Colunas ausentes: {missing_columns}"
        )

### Regra 2  

Email deve não existir na Silver de clientes  

Email duplicado entre lotes diferentes indica cliente recadastrado.


In [0]:
def validate_email_exists_in_silver(
    df_snapshot: DataFrame,
    df_silver: DataFrame
) -> tuple[DataFrame, DataFrame]:
    """
    Remove emails já existentes
    na Silver.

    Args:
        df_snapshot (DataFrame):
            Micro-lote.

        df_silver (DataFrame):
            Dados atuais da Silver.

    Returns:
        tuple:
            (valid_df, invalid_df)
    """

    invalid_df = (
        df_snapshot.join(
            df_silver.select("email").distinct(),
            on="email",
            how="inner"
        )
    )

    valid_df = (
        df_snapshot.join(
            df_silver.select("email").distinct(),
            on="email",
            how="left_anti"
        )
    )

    invalid_df = add_quarantine_metadata(
        invalid_df,
        "EMAIL_JA_EXISTE_SILVER"
    )

    return valid_df, invalid_df

### Regra 3  

dt_cadastro deve ser parseável como datetime e não pode ser futura 

Data futura é dado corrompido.


In [0]:
def validate_dt_cadastro(
    df: DataFrame
) -> tuple[DataFrame, DataFrame]:
    """
    Valida dt_cadastro.

    Args:
        df (DataFrame):
            DataFrame Spark.

    Returns:
        tuple:
            (valid_df, invalid_df)
    """

    valid_df = (
        df.filter(
            df.dt_cadastro <= current_timestamp()
        )
    )

    invalid_df = (
        df.filter(
            df.dt_cadastro > current_timestamp()
        )
    )

    invalid_df = add_quarantine_metadata(
        invalid_df,
        "DATA_CADASTRO_FUTURA"
    )

    return valid_df, invalid_df

### Regra 4  

uuid_cliente deve ter formato UUID v4 válido (32 hex + hífens) 

UUID malformado quebra integrações com microserviços externos.


In [0]:
def validate_uuid(
    df: DataFrame
) -> tuple[DataFrame, DataFrame]:
    """
    Valida UUID v4.

    Args:
        df (DataFrame):
            DataFrame Spark.

    Returns:
        tuple:
            (valid_df, invalid_df)
    """

    uuid_regex = (
        r'^[0-9a-fA-F]{8}-'
        r'[0-9a-fA-F]{4}-'
        r'4[0-9a-fA-F]{3}-'
        r'[89abAB][0-9a-fA-F]{3}-'
        r'[0-9a-fA-F]{12}$'
    )

    valid_df = (
        df.filter(
            col("uuid_cliente").rlike(uuid_regex)
        )
    )

    invalid_df = (
        df.filter(
            ~col("uuid_cliente").rlike(uuid_regex)
        )
    )

    invalid_df = add_quarantine_metadata(
        invalid_df,
        "UUID_INVALIDO"
    )

    return valid_df, invalid_df

### Regra 5  

Deduplicação de email dentro do micro-lote
	 
Lote pode reenviar o mesmo cliente.


In [0]:
def validate_duplicate_email(
    df: DataFrame
) -> tuple[DataFrame, DataFrame]:
    """
    Identifica emails duplicados
    dentro do micro-lote.

    Mantém o registro mais recente
    considerando dt_ultima_atualizacao.
    Caso seja nula, utiliza dt_cadastro.

    Args:
        df (DataFrame):
            DataFrame Spark.

    Returns:
        tuple:
            (valid_df, invalid_df)
    """

    window_spec = (
        Window
        .partitionBy("email")
        .orderBy(
            desc(
                coalesce(
                    col("dt_ultima_atualizacao"),
                    col("dt_cadastro")
                )
            )
        )
    )

    df_ranked = (
        df.withColumn(
            "row_num",
            row_number().over(window_spec)
        )
    )

    valid_df = (
        df_ranked
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

    invalid_df = (
        df_ranked
        .filter(col("row_num") > 1)
        .drop("row_num")
    )

    invalid_df = add_quarantine_metadata(
        invalid_df,
        "EMAIL_DUPLICADO_LOTE"
    )

    return valid_df, invalid_df

### Regra 5  
Deduplicação de id_cliente dentro do micro-lote	

Lote pode reenviar o mesmo cliente.

In [0]:
def validate_duplicate_id_cliente(
    df: DataFrame
) -> tuple[DataFrame, DataFrame]:
    """
    Identifica registros com id_cliente duplicado.

    Mantém o registro mais recente
    considerando dt_ultima_atualizacao.
    Caso seja nula, utiliza dt_cadastro.

    Args:
        df (DataFrame):
            DataFrame Spark.

    Returns:
        tuple:
            (valid_df, invalid_df)
    """

    window_spec = (
        Window
        .partitionBy("id_cliente")
        .orderBy(
            desc(
                coalesce(
                    col("dt_ultima_atualizacao"),
                    col("dt_cadastro")
                )
            )
        )
    )

    df_ranked = (
        df.withColumn(
            "row_num",
            row_number().over(window_spec)
        )
    )

    valid_df = (
        df_ranked
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

    invalid_df = (
        df_ranked
        .filter(col("row_num") > 1)
        .drop("row_num")
    )

    invalid_df = add_quarantine_metadata(
        invalid_df,
        "ID_CLIENTE_DUPLICADO"
    )

    return valid_df, invalid_df

### Regra 10  

Alertar se dt_ultima_atualizacao < dt_cadastro  

Impossibilidade lógica: atualização antes do cadastro.

In [0]:
def validate_update_date(
    df: DataFrame
) -> tuple[DataFrame, DataFrame]:
    """
    Valida consistência temporal.

    Args:
        df (DataFrame):
            DataFrame Spark.

    Returns:
        tuple:
            (valid_df, invalid_df)
    """

    valid_df = (
        df.filter(
            (col("dt_ultima_atualizacao").isNull())
            |
            (
                col("dt_ultima_atualizacao")
                >=
                col("dt_cadastro")
            )
        )
    )

    invalid_df = (
        df.filter(
            col("dt_ultima_atualizacao")
            <
            col("dt_cadastro")
        )
    )

    invalid_df = add_quarantine_metadata(
        invalid_df,
        "DATA_ATUALIZACAO_INVALIDA"
    )

    return valid_df, invalid_df

## Validação das regras da GOLD 'clientes'

### Regra 6  

KPI: Número de novos clientes cadastrados nos últimos 10 minutos  

Pico de cadastros pode indicar campanha de marketing ou bot. Monitorar em tempo real.


In [0]:
def create_kpi_new_customers_last_10min(
    df: DataFrame
) -> DataFrame:
    """
    Calcula a quantidade de novos clientes
    cadastrados nos últimos 10 minutos.

    Args:
        df (DataFrame):
            DataFrame Silver de clientes.

    Returns:
        DataFrame:
            KPI consolidado.
    """

    df_kpi = (
        df
        .filter(
            col("dt_cadastro")
            >= expr(
                "current_timestamp() - interval 10 minutes"
            )
        )
        .agg(
            count("*").alias(
                "new_customers_last_10min"
            )
        )
        .withColumn(
            "gold_processed_at",
            current_timestamp()
        )
    )

    return df_kpi

### Regra 7 

KPI: Taxa de cadastros por provedor de email (gmail, yahoo, hotmail…) em tempo real 

Domínio desconhecido em alta % pode indicar cadastros de bots com emails gerados.

In [0]:
def create_kpi_email_provider_rate(
    df: DataFrame
) -> DataFrame:
    """
    Calcula a distribuição de clientes
    por provedor de email.

    Args:
        df (DataFrame):
            DataFrame Silver de clientes.

    Returns:
        DataFrame:
            KPI por domínio de email.
    """
    
    total_customers = df.count()

    # Tratativa para evitar divisão por zero caso o DataFrame esteja vazio
    if total_customers == 0:
        total_customers = 1

    df_kpi = (
        df
        .withColumn(
            "email_provider",
            split(
                col("email"),
                "@"
            ).getItem(1)
        )
        .groupBy(
            "email_provider"
        )
        .agg(
            count("*").alias(
                "customer_count"
            )
        )
        .withColumn(
            "customer_rate",
            round(
                col("customer_count")
                / lit(total_customers)  # Usando F.lit corretamente
                * 100,
                2
            )
        )
        .withColumn(
            "gold_processed_at",
            current_timestamp()
        )
    )

    return df_kpi

### Regra 9  

KPI: Tempo médio entre dt_cadastro e primeira compra (quando disponível via join)  

Mede efetividade do funil. Se o tempo aumentar, pode indicar problema no onboarding.

In [0]:
def create_kpi_avg_time_to_first_purchase(
    df_customers: DataFrame
) -> DataFrame:
    """
    Calcula o tempo médio entre
    cadastro e primeira compra.

    Args:
        df_customers (DataFrame):
            Silver de clientes.

    Returns:
        DataFrame:
            KPI consolidado.
    """
    df_orders = read_delta_to_spark_df(
        path=f"abfs://{container_name_data_lake}/silver/ecommerce_pedidos",
        storage_options=storage_options
        )
    
    df_first_purchase = (
        df_orders
        .groupBy(
            "id_cliente"
        )
        .agg(
            min(
                "dt_pedido"
            ).alias(
                "dt_primeira_compra"
            )
        )
    )

    df_join = (
        df_customers
        .join(
            df_first_purchase,
            on="id_cliente",
            how="inner"
        )
    )

    df_join = (
        df_join
        .withColumn(
            "days_to_first_purchase",
            datediff(
                col("dt_primeira_compra"),
                col("dt_cadastro")
            )
        )
    )

    df_kpi = (
        df_join
        .agg(
            avg(
                "days_to_first_purchase"
            ).alias(
                "avg_days_to_first_purchase"
            )
        )
        .withColumn(
            "gold_processed_at",
            current_timestamp()
        )
    )

    return df_kpi

## Validação das regras da SILVER 'enderecos'

### Regra 1  
Schema completo presente (id_endereco, id_cliente, logradouro, cep, cidade, estado, is_principal)

Verificação de schema no lote de endereços.

In [0]:
def validate_enderecos_schema(
    df: DataFrame
) -> None:
    """
    Valida se todas as colunas obrigatórias
    existem no micro-lote de endereços.

    Args:
        df (DataFrame):
            DataFrame Spark.

    Raises:
        ValueError:
            Caso alguma coluna obrigatória esteja ausente.
    """

    required_columns = [
        "id_endereco",
        "id_cliente",
        "logradouro",
        "cep",
        "cidade",
        "estado",
        "is_principal"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Schema inválido. Colunas ausentes: {missing_columns}"
        )

### Regra 2
Schema completo presente (id_endereco, id_cliente, logradouro, cep, cidade, estado, is_principal)

Verificação de schema no lote de endereços.

In [0]:
def validate_estado_uf(
    df: DataFrame
) -> tuple[DataFrame, DataFrame]:
    """
    Valida se a coluna estado contém uma UF brasileira válida.

    Args:
        df (DataFrame):
            DataFrame Spark.

    Returns:
        tuple:
            (valid_df, invalid_df)
    """

    valid_ufs = [
        "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES",
        "GO", "MA", "MT", "MS", "MG", "PA", "PB", "PR",
        "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC",
        "SP", "SE", "TO"
    ]

    df = df.withColumn(
        "estado",
        upper(trim(col("estado")))
    )

    valid_df = (
        df.filter(
            col("estado").isin(valid_ufs)
        )
    )

    invalid_df = (
        df.filter(
            ~col("estado").isin(valid_ufs)
        )
    )

    invalid_df = add_quarantine_metadata(
        invalid_df,
        "ESTADO_INVALIDO"
    )

    return valid_df, invalid_df

### Regra 3
cep deve ter 8 dígitos numéricos

CEP é chave de roteamento logístico.

In [0]:
def validate_cep(
    df: DataFrame
) -> tuple[DataFrame, DataFrame]:

    df = df.withColumn(
        "cep",
        regexp_replace(
            col("cep").cast("string"),
            "[^0-9]",
            ""
        )
    )

    valid_condition = (
        col("cep").isNotNull()
        & col("cep").rlike("^[0-9]{8}$")
    )

    valid_df = df.filter(valid_condition)

    invalid_df = df.filter(~valid_condition)

    invalid_df = add_quarantine_metadata(
        invalid_df,
        "CEP_INVALIDO"
    )

    return valid_df, invalid_df

## Validação das regras da GOLD 'enderecos'

### Regra 4
KPI: Distribuição geográfica de novos endereços por estado em tempo real

Detectar se um estado está com volume anormal de cadastros (possível campanha regional).

In [0]:
def create_kpi_address_distribution_by_state(
    df: DataFrame
) -> DataFrame:
    """
    Calcula a distribuição de endereços por estado.

    Args:
        df (DataFrame):
            DataFrame Silver de endereços.

    Returns:
        DataFrame:
            KPI com quantidade e percentual de endereços por estado.
    """

    total_addresses = df.count()

    df_kpi = (
        df
        .groupBy("estado")
        .agg(
            count("*").alias("address_count")
        )
        .withColumn(
            "address_rate",
            round(
                col("address_count") / lit(total_addresses) * 100,
                2
            )
        )
    )

    return df_kpi

# DEFINIÇÃO DA UTILIZAÇÃO DAS REGRAS DE TRANSFORMAÇÃO DA SILVER

## Definir quais regras serão aplicadas na SILVER de acordo com a entity_name


In [0]:
def apply_silver_validation_rules(
    entity_name: str,
    df_snapshot: DataFrame,
    path_target: str,
    storage_options : dict,
) -> tuple[DataFrame, DataFrame]:
    """
    Aplica as regras de validação da camada Silver
    conforme a entidade processada.

    A leitura da Silver é feita apenas para entidades
    cujas regras dependem do histórico já gravado.
    """

    # ----------------------
    # Entidade: Clientes
    # ----------------------

    if entity_name == "clientes":

        try:
            log.info("Lendo Silver para validação histórica")

            df_silver = read_delta_to_spark_df(
                path=path_target,
                storage_options=storage_options
            )

            log.info("Silver lida com sucesso")

        except Exception:

            log.info(
                "Silver ainda não existe. "
                "Criando DataFrame vazio."
            )

            df_silver = spark.createDataFrame(
                [],
                df_snapshot.schema
            )

        return apply_silver_validation_rules_clientes(
            df_snapshot=df_snapshot,
            df_silver=df_silver
        )

    # ----------------------
    # Entidade: Endereços
    # ----------------------

    elif entity_name == "enderecos":

        return apply_silver_validation_rules_enderecos(
            df_snapshot=df_snapshot
        )

    # ----------------------
    # Entidade não suportada
    # ----------------------

    else:

        raise ValueError(
            f"Entidade não suportada: {entity_name}"
        )

## Aplicar todas as regras de transformação da Silver 'clientes'

In [0]:
def apply_silver_validation_rules_clientes(
    df_snapshot: DataFrame,
    df_silver: DataFrame
) -> tuple[DataFrame, DataFrame]:
    """
    Aplica as regras de validação da camada Silver
    para a entidade ecommerce_clientes.

    Args:
        df_snapshot (DataFrame):
            Micro-lote de clientes vindo da Bronze.

        df_silver (DataFrame):
            DataFrame Silver atual de clientes,
            usado para validar emails já existentes.

    Returns:
        tuple:
            (df_valid, df_quarantine)
    """

    # ----------------------
    # Regra 1 - Schema
    # ----------------------

    validate_clientes_schema(
        df_snapshot
    )

    df_quarantine = None

    # ----------------------
    # Regra 2 - email não existir na Silver
    # ----------------------

    log.info("Validando Regra 2")

    df_snapshot, df_invalid = (
        validate_email_exists_in_silver(
            df_snapshot=df_snapshot,
            df_silver=df_silver
        )
    )

    df_quarantine = append_to_quarantine(
        df_quarantine,
        df_invalid
    )

    # ----------------------
    # Regra 3 - dt_cadastro não pode ser data futura
    # ----------------------

    log.info("Validando Regra 3")

    df_snapshot, df_invalid = (
        validate_dt_cadastro(
            df_snapshot
        )
    )

    df_quarantine = append_to_quarantine(
        df_quarantine,
        df_invalid
    )

    # ----------------------
    # Regra 4 - UUID não pode ser inválido
    # ----------------------

    log.info("Validando Regra 4")

    df_snapshot, df_invalid = (
        validate_uuid(
            df_snapshot
        )
    )

    df_quarantine = append_to_quarantine(
        df_quarantine,
        df_invalid
    )

    # ----------------------
    # Regra 5 - id_cliente não pode ser duplicado
    # ----------------------

    log.info("Validando Regra 5 - Cliente")

    df_snapshot, df_invalid = (
        validate_duplicate_id_cliente(
            df_snapshot
        )
    )

    df_quarantine = append_to_quarantine(
        df_quarantine,
        df_invalid
    )

    # ----------------------
    # Regra 5 - email não pode ser duplicado
    # ----------------------

    log.info("Validando Regra 5 - Email")

    df_snapshot, df_invalid = (
        validate_duplicate_email(
            df_snapshot
        )
    )

    df_quarantine = append_to_quarantine(
        df_quarantine,
        df_invalid
    )

    # ----------------------
    # Regra 10 - dt_ultima_atualizacao >= dt_cadastro
    # ----------------------

    log.info(
        "Validando Regra 10 - "
        "dt_ultima_atualizacao >= dt_cadastro"
    )

    df_snapshot, df_invalid = (
        validate_update_date(
            df_snapshot
        )
    )

    df_quarantine = append_to_quarantine(
        df_quarantine,
        df_invalid
    )

    return df_snapshot, df_quarantine

## Aplicar todas as regras de transformação da Silver 'enderecos'

In [0]:
def apply_silver_validation_rules_enderecos(
    df_snapshot: DataFrame
) -> tuple[DataFrame, DataFrame]:
    
    """
    Aplica as regras de validação da camada Silver
    para a entidade ecommerce_enderecos.

    Args:
        df_snapshot (DataFrame):
            Micro-lote de endereços vindo da Bronze.

    Returns:
        tuple:
            (df_valid, df_quarantine)
    """

    # ----------------------
    # Regra 1 - Schema
    # ----------------------

    validate_enderecos_schema(
        df_snapshot
    )

    df_quarantine = None

    # ----------------------
    # Regra 2 - UF válida
    # ----------------------

    log.info("Validando Regra 2")

    df_snapshot, df_invalid = validate_estado_uf(
        df=df_snapshot
    )

    df_quarantine = append_to_quarantine(
        df_quarantine,
        df_invalid
    )

    log.info(
        f"{df_snapshot.count()} registros válidos | "
        f"{df_invalid.count()} registros inválidos"
    )

    # ----------------------
    # Regra 3 - CEP 8 dígitos numéricos
    # ----------------------

    log.info("Validando Regra 3")

    df_snapshot, df_invalid = validate_cep(
        df_snapshot
    )

    df_quarantine = append_to_quarantine(
        df_quarantine,
        df_invalid
    )

    log.info(
        f"{df_snapshot.count()} registros válidos | "
        f"{df_invalid.count()} registros inválidos"
    )

    return df_snapshot, df_quarantine

# DEFINIÇÃO DA UTILIZAÇÃO DAS REGRAS DE TRANSFORMAÇÃO DA GOLD

## Define quais regras serão aplicadas na GOLD de acordo com a entity_name


In [0]:
def apply_gold_validation_rules(
    entity_name: str,
    df_silver: DataFrame
) -> None:
    """
    Aplica as regras da camada Gold
    conforme a entidade processada.

    Args:
        entity_name (str):
            Nome da entidade.
            Ex.:
            clientes, enderecos.

        df_silver (DataFrame):
            DataFrame da camada Silver
            utilizado como origem para
            geração dos KPIs da Gold.

    Returns:
        None
    """

    # ----------------------
    # Entidade: Clientes
    # ----------------------

    if entity_name == "clientes":

        apply_gold_validation_rules_clientes(
            df_silver=df_silver,
            path_gold_base=path_gold_base,
            storage_options=storage_options
        )

    # ----------------------
    # Entidade: Endereços
    # ----------------------

    elif entity_name == "enderecos":

        apply_gold_validation_rules_enderecos(
            df_silver=df_silver,
            path_gold_base=path_gold_base,
            storage_options=storage_options
        )

    # ----------------------
    # Entidade não suportada
    # ----------------------

    else:

        raise ValueError(
            f"Entidade não suportada: "
            f"{entity_name}"
        )

## Aplica todas as regras de transformação da Gold 'clientes'

In [0]:

def apply_gold_validation_rules_clientes(
    df_silver: DataFrame,
    path_gold_base: str,
    storage_options : dict
) -> None:
    """
    Aplica e grava as regras da camada Gold
    para a entidade ecommerce_clientes.
    """

    partition_by = [
        "processed_year",
        "processed_month",
        "processed_day",
        "processed_hour"
    ]

    # ----------------------
    # Regra 6
    # ----------------------

    log.info("Calculando KPI 6")

    path_gold_new_customers = (
        f"{path_gold_base}/new_customers"
    )

    df_kpi_new_customers = add_gold_audit_columns(
        create_kpi_new_customers_last_10min(
            df_silver
        )
    )

    write_adls_partitioned(
        df=df_kpi_new_customers,
        path=path_gold_new_customers,
        storage_options=storage_options,
        partition_by=partition_by,
        mode="append"
    )

    log.info("Regra 6 gravada com sucesso.")

    # ----------------------
    # Regra 7
    # ----------------------

    log.info("Calculando KPI 7")

    path_gold_email_provider = (
        f"{path_gold_base}/email_provider_rate"
    )

    df_kpi_email_provider = add_gold_audit_columns(
        create_kpi_email_provider_rate(
            df_silver
        )
    )

    write_adls_partitioned(
        df=df_kpi_email_provider,
        path=path_gold_email_provider,
        storage_options=storage_options,
        partition_by=partition_by,
        mode="overwrite"
    )

    log.info("Regra 7 gravada com sucesso.")

    # ----------------------
    # Regra 9
    # ----------------------

    log.info("Calculando KPI 9")

    path_gold_first_purchase = (
        f"{path_gold_base}/first_purchase"
    )

    df_kpi_first_purchase = add_gold_audit_columns(
        create_kpi_avg_time_to_first_purchase(
            df_silver
        )
    )

    write_adls_partitioned(
        df=df_kpi_first_purchase,
        path=path_gold_first_purchase,
        storage_options=storage_options,
        partition_by=partition_by,
        mode="overwrite"
    )

    log.info("Regra 9 gravada com sucesso.")

## Aplica todas as regras de transformação da Gold 'enderecos'

In [0]:
def apply_gold_validation_rules_enderecos(
    df_silver: DataFrame,
    path_gold_base: str,
    storage_options,
) -> None:
    """
    Aplica e grava as regras da camada Gold
    para a entidade ecommerce_enderecos.
    """

    partition_by = [
        "processed_year",
        "processed_month",
        "processed_day",
        "processed_hour"
    ]

    # ----------------------
    # Regra 4
    # ----------------------

    log.info(
        "Calculando Regra 4 - Distribuição por estado"
    )

    path_kpi_address_distribution = (
        f"{path_gold_base}/address_distribution_by_state"
    )

    df_kpi_address_distribution = add_gold_audit_columns(
        create_kpi_address_distribution_by_state(
            df_silver
        )
    )

    write_adls_partitioned(
        df=df_kpi_address_distribution,
        path=path_kpi_address_distribution,
        storage_options=storage_options,
        partition_by=partition_by,
        mode="overwrite"
    )

    log.info("Regra 4 gravada com sucesso.")